# ISO18571 Quickstart

This notebook is a guided path through three useful example datasets: the bundled demo CSVs, one downloaded official Annex case, and one deterministic generated signal.

Every curve passed to `ISO18571` is an `(n, 2)` array. Column 0 is time, column 1 is signal value. The reference and comparison time columns must be finite, strictly increasing, uniformly spaced, and equal to each other.

The scorer returns raw component values, rounded ISO-style component ratings, and phase-alignment metadata. Use `phase_reference_start`, `phase_comparison_start`, and `phase_shift_length` to slice aligned signal sections for visual inspection.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "tools").is_dir() and (REPO_ROOT.parent / "tools").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from iso18571 import ISO18571, backend_info
from tools import signals
from tools.annex import AnnexDataset
from tools.signals import SignalGenerator

backend_info()

The setup cell resolves the repository root before importing `tools.annex` and `tools.signals`, so the notebook works whether it is launched from the repository root or from the `examples/` directory.

The small helpers below keep the notebook focused on the workflow. `load_curve` mirrors the CSV-loading pattern in `main.py`, `plot_curve_pair` gives quick signal visibility, and `score_summary` collects the values users usually inspect first.

In [ ]:
def load_curve(path: Path) -> np.ndarray:
    curve = np.loadtxt(path, delimiter=",")
    if curve.ndim != 2 or curve.shape[1] != 2:
        raise ValueError(f"{path} must contain exactly two columns: time,value")
    return curve


def plot_curve_pair(
    reference_curve: np.ndarray, comparison_curve: np.ndarray, title: str
) -> None:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(reference_curve[:, 0], reference_curve[:, 1], label="reference")
    ax.plot(comparison_curve[:, 0], comparison_curve[:, 1], label="comparison")
    ax.set_title(title)
    ax.set_xlabel("time")
    ax.set_ylabel("signal")
    ax.grid(True, alpha=0.25)
    ax.legend()
    plt.show()


def aligned_curves(
    score: ISO18571, reference_curve: np.ndarray, comparison_curve: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    scores = score.scores
    reference_start = int(scores["phase_reference_start"])
    comparison_start = int(scores["phase_comparison_start"])
    shift_length = int(scores["phase_shift_length"])
    reference_stop = reference_start + shift_length
    comparison_stop = comparison_start + shift_length
    return (
        reference_curve[reference_start:reference_stop],
        comparison_curve[comparison_start:comparison_stop],
    )


def score_summary(score: ISO18571) -> dict[str, object]:
    scores = score.scores
    return {
        "rounded_scores": {
            "R": score.overall_rating(),
            "Z": score.corridor_rating(),
            "EP": score.phase_rating(),
            "EM": score.magnitude_rating(),
            "ES": score.slope_rating(),
        },
        "phase_n_eps": score.phase_n_eps,
        "phase_rho_e": score.phase_rho_e,
        "phase_reference_start": int(scores["phase_reference_start"]),
        "phase_comparison_start": int(scores["phase_comparison_start"]),
        "phase_shift_length": int(scores["phase_shift_length"]),
    }

## Stage 1: bundled demo CSVs

`examples/reference.csv` and `examples/comparison.csv` are no-header `time,value` files checked into the repository. This is the same pair used by the no-argument `main.py` script.

In [ ]:
reference_csv = REPO_ROOT / "examples/reference.csv"
comparison_csv = REPO_ROOT / "examples/comparison.csv"

reference_curve = load_curve(reference_csv)
comparison_curve = load_curve(comparison_csv)

reference_curve.shape, comparison_curve.shape

Before scoring, plot the two input signals. Visual inspection is useful because ISO/TS 18571 combines corridor, phase, magnitude, and slope behavior; a single overall number should not be the only thing you look at.

In [ ]:
plot_curve_pair(reference_curve, comparison_curve, "Bundled example signals")

Now run the scorer. The `scores` dictionary contains unrounded component values and phase-alignment metadata from the native backend.

In [ ]:
demo_score = ISO18571(reference_curve, comparison_curve)
demo_score.scores

The summary below shows rounded ratings for `R`, `Z`, `EP`, `EM`, and `ES`, plus the phase alignment values `phase_n_eps`, `phase_rho_e`, and the aligned-slice indices.

In [ ]:
score_summary(demo_score)

The aligned slices are the signal sections used for magnitude and slope scoring. Plotting them helps separate phase differences from value-shape differences.

In [ ]:
aligned_reference_curve, aligned_comparison_curve = aligned_curves(
    demo_score, reference_curve, comparison_curve
)
plot_curve_pair(
    aligned_reference_curve,
    aligned_comparison_curve,
    "Bundled example aligned signals",
)

The native scorer treats the time column as authoritative. This cell intentionally perturbs one comparison timestamp to show that inconsistent time grids are rejected instead of silently resampled.

In [ ]:
bad_comparison = comparison_curve.copy()
bad_comparison[20, 0] += 0.5 * (comparison_curve[1, 0] - comparison_curve[0, 0])

try:
    ISO18571(reference_curve, bad_comparison)
except ValueError as exc:
    print(f"{type(exc).__name__}: {exc}")

## Stage 2: downloaded official Annex CSVs

The official ISO/TS 18571 Annex CSV ZIP is downloaded on demand, hash-checked, and extracted under `examples/data/annex/official`. That directory is ignored by git because these files are reproducible external data.

In [ ]:
official_annex_dir = REPO_ROOT / "examples/data/annex/official"
official_dataset = AnnexDataset.ensure(official_annex_dir)
official_case = next(official_dataset.cases())
official_reference_curve = official_case.reference_curve()
official_comparison_curve = official_case.comparison_curve()

official_score = ISO18571(official_reference_curve, official_comparison_curve)
official_case.name, score_summary(official_score)

The selected Annex case uses the official `Test` signal as reference and the official `CAE` signal as comparison. The plot makes it easy to connect the score components back to the signal behavior.

In [ ]:
plot_curve_pair(
    official_reference_curve,
    official_comparison_curve,
    f"Official Annex: {official_case.name}",
)

## Stage 3: generated synthetic signal

This generated case is built in memory with `tools.signals.SignalGenerator`. It is deterministic, uses the same `(n, 2)` curve contract as file-backed examples, and avoids relying on removed example-data scaffolding.

In [ ]:
generated_name = "synthetic_sine_chirp_noise"
generated_reference_curve = (
    SignalGenerator(512, 0.001, seed=18571)
    .add(signals.sine, frequency=6.0, amplitude=1.0)
    .add(signals.chirp, start_frequency=1.0, end_frequency=9.0, amplitude=0.2)
    .add(signals.gaussian_noise, std=0.02)
    .curve()
)
generated_comparison_curve = (
    SignalGenerator(512, 0.001, seed=18571)
    .add(signals.sine, frequency=6.0, amplitude=0.96, sample_shift=4)
    .add(
        signals.chirp,
        start_frequency=1.0,
        end_frequency=9.0,
        amplitude=0.18,
        sample_shift=4,
    )
    .add(signals.gaussian_noise, std=0.02, offset=0.03)
    .curve()
)

generated_score = ISO18571(generated_reference_curve, generated_comparison_curve)
generated_name, generated_reference_curve.shape, score_summary(generated_score)

The generated signal combines smooth periodic structure, a chirp, noise, phase shift, and a small offset. The test suite covers a broader set of generated signal families.

In [ ]:
plot_curve_pair(
    generated_reference_curve,
    generated_comparison_curve,
    f"Generated signal: {generated_name}",
)